Download do Dataset:

In [ ]:
import requests

url = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG/2026/INFLUD26-19-10-2026.parquet"
local_filename = "../data/srag_data_2026.parquet"

# download e escrita
response = requests.get(url, allow_redirects=True)
response.raise_for_status()  # HTTP errors

with open(local_filename, "wb") as file:
    file.write(response.content)

print("Download completo.")


Carregamento dos dados:

In [ ]:
import pandas as pd
local_filename = "../data/srag_data_2026.parquet"
df = pd.read_parquet(path=local_filename)

Descrição do DataFrame:

In [ ]:
df.shape

Temos ~232 mil registros e 194 colunas. Já no `describe`:

In [ ]:
df.describe()

Vemos que somente as colunas de datas são numéricas, o que é claro dos dicionários de dados [2019 a 2025- Dicionário de Dados](https://dadosabertos.saude.gov.br/dataset/srag-2019-a-2026/resource/3135ac9c-2019-4989-a893-2ed50ebd8e68) onde vemos ou datas ou caracteres:

In [ ]:
df.info(verbose=True)

Vemos um grande dataset ~400 MB, logo será útil para nós usarmos somente as colunas necessárias. Com base no dicionário de dados vemos quee as colunas úteis são:
- NU_NOTIFIC: Número do registro
- DT_NOTIFIC: Data de preenchimento da ficha de notificação.
- DT_SIN_PRI: Data de 1º sintomas do caso.
- DT_UT_DOSE: Data da última dose de vacina contra gripe que o paciente tomou.
- DT_INTERNA: Data em que o paciente foi hospitalizado.
- DT_EVOLUCA: Data da alta ou óbito.
- EVOLUCAO: Evolução do caso (1-Cura / 2-Óbito / 3- Óbito por outras causas / 9-Ignorado).
- VACINA_COV: Informar se o paciente recebeu vacina COVID-19, após verificar a documentação/caderneta (1-Sim / 2-Não / 9-Ignorado).

Para calcularmos as métricas:
- [X] Taxa de aumento de casos;
- [] Taxa de mortalidade;
- [] Taxa de ocupação de UTI;
- [X] Taxa de vacinação da população.
- [X] Número diário de casos dos últimos 30 dias.
- [X] Número mensal de casos dos últimos 12 meses

In [ ]:
cols = ["NU_NOTIFIC", "DT_NOTIFIC", "DT_SIN_PRI", "DT_UT_DOSE", "DT_INTERNA", "DT_EVOLUCA", "EVOLUCAO", "VACINA_COV"]
df_reduced = df[cols]
df_reduced.head(10)

In [ ]:
df_reduced.dropna(subset=["EVOLUCAO"])

In [ ]:
evolucao = {"1":"cura", "2":"obito", "3":"obitos_outros"}
df_reduced["EVOLUCAO"] = df_reduced["EVOLUCAO"].map(evolucao)

In [ ]:
df_reduced

In [ ]:
num_casos_notificado = df.groupby('DT_NOTIFIC')['NU_NOTIFIC'].count()
num_casos_notificado

In [ ]:
type(num_casos_notificado)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="darkgrid")  # Optional style enhancement
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_notificado, marker="o")

# 4. Final touches with Matplotlib
plt.title("Single Time Series Tracking")
plt.xlabel("Data")
plt.ylabel("Número casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="darkgrid")  # Optional style enhancement
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_notificado.rolling(window=7).mean(), marker="o")

# 4. Final touches with Matplotlib
plt.title("Single Time Series Tracking")
plt.xlabel("Data")
plt.ylabel("Número casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="darkgrid")  # Optional style enhancement
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_notificado.rolling(window=14).mean(), marker="o")

# 4. Final touches with Matplotlib
plt.title("Single Time Series Tracking")
plt.xlabel("Data")
plt.ylabel("Número casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="darkgrid")  # Optional style enhancement
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_notificado.rolling(window=30).mean(), marker="o")

# 4. Final touches with Matplotlib
plt.title("Single Time Series Tracking")
plt.xlabel("Data")
plt.ylabel("Número casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="darkgrid")  # Optional style enhancement
plt.figure(figsize=(10, 5))

sns.lineplot(data=num_casos_notificado.tail(30), marker="o")

# 4. Final touches with Matplotlib
plt.title("Single Time Series Tracking")
plt.xlabel("Data")
plt.ylabel("Número casos")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
num_casos_rolling_12_m = num_casos_notificado.rolling(7).mean().tail(365).dropna()
num_casos_rolling_30_d = num_casos_notificado.rolling(7).mean().tail(30).dropna()

growth_num_casos_30_d = (num_casos_rolling_30_d.iloc[-1] - num_casos_rolling_30_d.iloc[0])/num_casos_rolling_30_d.iloc[0]
growth_num_casos_12_m = (num_casos_rolling_12_m.iloc[-1] - num_casos_rolling_12_m.iloc[0])/num_casos_rolling_12_m.iloc[0]

print("Crescimento número de casos (30 d):", growth_num_casos_30_d)
print("Crescimento número de casos (12 m):", growth_num_casos_12_m)

In [ ]:
num_casos_rolling_30_d.iloc[0], num_casos_rolling_30_d.iloc[-1]

In [ ]:
vacinados_clean = df_reduced.groupby("VACINA_COV", dropna=False).count()[["NU_NOTIFIC"]]
vacinados_clean["pct"] = vacinados_clean["NU_NOTIFIC"]/len(df_reduced)
vacinados_clean

In [ ]:
max_date = df_reduced["DT_NOTIFIC"].max()
initial_30_d = max_date - pd.Timedelta(days=30)
initial_12_m = max_date - pd.Timedelta(days=365)

df_reduced_30_d = df_reduced[df_reduced["DT_NOTIFIC"]>=initial_30_d]

df_reduced_12_m = df_reduced[df_reduced["DT_NOTIFIC"]>=initial_12_m]

In [ ]:
mortalidade_clean_30_d = df_reduced_30_d.groupby("EVOLUCAO", dropna=False).count()[["NU_NOTIFIC"]]
mortalidade_clean_30_d["pct"] = mortalidade_clean_30_d["NU_NOTIFIC"]/len(df_reduced_30_d)
mortalidade_clean_30_d

In [ ]:
mortalidade_clean_12_m = df_reduced_12_m.groupby("EVOLUCAO", dropna=False).count()[["NU_NOTIFIC"]]
mortalidade_clean_12_m["pct"] = mortalidade_clean_12_m["NU_NOTIFIC"]/len(df_reduced_12_m)
mortalidade_clean_12_m